# Hill-Climbing for a Pac‑Man–like Game (From First Principles)

_Generated: 2025-10-22T17:40:35.256402Z_

**Objective.** Implement a tiny Pac‑Man–like grid world and optimize a **parameterized reactive policy** using a simple **hill‑climbing** algorithm. The agent chooses among actions {Up,Right,Down,Left,Stay} with a softmax over features (e.g., distances to pellets/ghosts, wall proximity). We perturb the weight vector and accept a candidate if it **improves average score** across a batch of episodes.

**What’s inside**
- Minimal Pac‑like simulator (pellets, power pellets, ghosts, terminal conditions)
- Hand‑crafted features and softmax action selection
- First‑principles hill‑climbing (with noise scheduling and evaluation rolls)
- Plots: learning curve; episode animation; final policy rollouts
- Artifacts & one‑click download

## 0) Environment & Dependencies

In [ ]:
import sys, platform, subprocess
def _sh(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, stderr=subprocess.STDOUT, text=True)
    except Exception as e:
        out = f"[command failed] {e}"
    print(out)

print("Python:", sys.version)
print("Platform:", platform.platform())
print("In Colab:", "google.colab" in sys.modules)

!pip -q install --upgrade pip
!pip -q install numpy matplotlib

## 1) Imports, Reproducibility, and Utilities

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

np.set_printoptions(precision=4, suppress=True)
SEED = 7
rng = np.random.default_rng(SEED)

ACTIONS = np.array([[ -1,  0],  # Up
                    [  0,  1],  # Right
                    [  1,  0],  # Down
                    [  0, -1],  # Left
                    [  0,  0]]) # Stay
ACTION_NAMES = ["U","R","D","L","S"]

def manhattan(a, b):
    return np.abs(a[0]-b[0]) + np.abs(a[1]-b[1])

## 2) Maze Definition

We define a small fixed maze. 0=free, 1=wall. Pellets initially populate free cells unless blocked; a few power pellets are placed.

In [ ]:
@dataclass
class Maze:
    grid: np.ndarray  # 0 free, 1 wall
    start_pac: tuple
    start_ghosts: list
    power_pellets: list
    step_limit: int = 400

def make_maze():
    grid = np.array([
        [1,1,1,1,1,1,1,1,1,1,1,1,1,1],
        [1,0,0,0,1,0,0,0,0,0,1,0,0,1],
        [1,0,1,0,1,0,1,1,1,0,1,0,0,1],
        [1,0,1,0,0,0,0,0,1,0,0,0,0,1],
        [1,0,1,1,1,1,1,0,1,1,1,1,0,1],
        [1,0,0,0,0,0,1,0,0,0,0,1,0,1],
        [1,1,1,1,1,0,1,1,1,1,0,1,0,1],
        [1,0,0,0,1,0,0,0,0,1,0,0,0,1],
        [1,0,1,0,1,1,1,1,0,1,1,1,0,1],
        [1,0,0,0,0,0,0,0,0,0,0,0,0,1],
        [1,1,1,1,1,1,1,1,1,1,1,1,1,1],
    ], dtype=int)
    start_pac = (9,1)
    start_ghosts = [(1,12), (3,8)]
    power_pellets = [(1,1), (9,12)]
    return Maze(grid, start_pac, start_ghosts, power_pellets, step_limit=350)

MAZE = make_maze()
H, W = MAZE.grid.shape

## 3) Game State and Transition Dynamics

Rules (simplified):
- Pac collects pellets (+1) and power pellets (+10, enables **frightened** timer).
- If a ghost touches Pac while **not frightened** → terminal with penalty (−50).
- If **frightened**, Pac can eat a ghost (+20) and ghost respawns at its start.
- Small step penalty (−0.05) to encourage efficiency.
- Ghosts pursue Pac with a **noisy greedy** policy (20% random).

In [ ]:
@dataclass
class GameState:
    pac: tuple
    ghosts: list
    pellets: set
    power: set
    frightened_timer: int
    score: float
    t: int
    done: bool

STEP_REWARD = -0.05
PELLET_REWARD = 1.0
POWER_REWARD = 10.0
EAT_GHOST_REWARD = 20.0
DEATH_PENALTY = -50.0
FRIGHTENED_STEPS = 25  # duration

def init_state(maze: Maze):
    pellets = set()
    for i in range(H):
        for j in range(W):
            if maze.grid[i,j]==0:
                pellets.add((i,j))
    pellets.discard(maze.start_pac)
    for g in maze.start_ghosts: pellets.discard(g)
    power = set(maze.power_pellets)
    for p in power: pellets.discard(p)

    return GameState(
        pac=maze.start_pac,
        ghosts=list(maze.start_ghosts),
        pellets=pellets,
        power=set(power),
        frightened_timer=0,
        score=0.0,
        t=0,
        done=False
    )

def in_bounds(i,j): return 0 <= i < H and 0 <= j < W
def is_free(i,j): return in_bounds(i,j) and MAZE.grid[i,j]==0

def step_dir(pos, d):
    ni, nj = pos[0]+d[0], pos[1]+d[1]
    return (ni, nj) if is_free(ni,nj) else pos

def move_ghost(gpos, pac, noise=0.2):
    if np.random.rand() < noise:
        candidates = [step_dir(gpos, d) for d in ACTIONS]
        # Avoid being stationary too often
        if len(candidates) > 1:
            candidates = [c for c in candidates if c != gpos] or candidates
        return candidates[np.random.randint(len(candidates))]
    best = gpos; best_d = manhattan(gpos, pac)
    for d in ACTIONS:
        nxt = step_dir(gpos, d)
        dman = manhattan(nxt, pac)
        if dman < best_d:
            best = nxt; best_d = dman
    return best

def advance(state: GameState, a_idx: int, maze: Maze):
    if state.done: return state

    pac_next = step_dir(state.pac, ACTIONS[a_idx])
    score = state.score + STEP_REWARD
    frightened = max(0, state.frightened_timer-1)

    if pac_next in state.pellets:
        score += PELLET_REWARD
        state.pellets.remove(pac_next)
    if pac_next in state.power:
        score += POWER_REWARD
        state.power.remove(pac_next)
        frightened = FRIGHTENED_STEPS

    new_ghosts = []
    for k, g in enumerate(state.ghosts):
        g_next = move_ghost(g, pac_next, noise=0.2)
        if g_next == pac_next:
            if frightened > 0:
                score += EAT_GHOST_REWARD
                g_next = maze.start_ghosts[k]
            else:
                score += DEATH_PENALTY
                return GameState(pac_next, new_ghosts+[g_next], state.pellets, state.power,
                                 frightened, score, state.t+1, True)
        new_ghosts.append(g_next)

    if pac_next in new_ghosts and frightened == 0:
        score += DEATH_PENALTY
        return GameState(pac_next, new_ghosts, state.pellets, state.power,
                         frightened, score, state.t+1, True)

    done = (state.t+1 >= maze.step_limit) or (len(state.pellets)==0 and len(state.power)==0)
    return GameState(pac_next, new_ghosts, state.pellets, state.power,
                     frightened, score, state.t+1, done)

## 4) Feature Map and Stochastic Policy

We define features $\phi(s,a)$ that favor moving towards pellets and away from ghosts, avoiding walls, etc. The policy is softmax: $\pi(a\mid s;w) \propto \exp(w^\top\phi(s,a))$. Action is sampled each step.

In [ ]:
def manhattan(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def nearest(point, targets):
    if not targets: return np.inf
    return min(manhattan(point, t) for t in targets)

def features(state: GameState, a_idx: int):
    nxt = step_dir(state.pac, ACTIONS[a_idx])
    d_pel = nearest(nxt, state.pellets) if state.pellets else 0.0
    d_pow = nearest(nxt, state.power) if state.power else 0.0
    d_gh  = nearest(nxt, state.ghosts) if state.ghosts else np.inf
    wall_u = 1.0 if not is_free(nxt[0]-1, nxt[1]) else 0.0
    wall_r = 1.0 if not is_free(nxt[0], nxt[1]+1) else 0.0
    wall_d = 1.0 if not is_free(nxt[0]+1, nxt[1]) else 0.0
    wall_l = 1.0 if not is_free(nxt[0], nxt[1]-1) else 0.0
    walls = wall_u + wall_r + wall_d + wall_l

    f = np.array([
        1.0,
        -d_pel/10.0,
        -d_pow/10.0,
        +np.clip(10.0/d_gh if d_gh>0 else 10.0, 0, 10.0),
        -walls,
        1.0 if nxt in state.pellets else 0.0,
        1.0 if nxt in state.power else 0.0,
        -1.0 if nxt == state.pac else 0.0
    ], dtype=float)
    if state.frightened_timer > 0:
        f[3] *= -1.0
    return f

FEAT_DIM = len(features(init_state(MAZE), 0))

def policy_logits(state, w):
    Phi = np.stack([features(state, a) for a in range(len(ACTIONS))], axis=0)
    return Phi @ w, Phi

def softmax(z):
    z = z - np.max(z)
    e = np.exp(np.clip(z, -40, 40))
    return e / np.sum(e)

def sample_action(state, w):
    logits, _ = policy_logits(state, w)
    p = softmax(logits)
    return int(np.random.choice(len(ACTIONS), p=p)), p

## 5) Rollouts and Evaluation

In [ ]:
def run_episode(w, max_steps=None, seed=None, record=False):
    if seed is not None:
        rs = np.random.RandomState(seed)
        np.random.set_state(rs.get_state())
    s = init_state(MAZE)
    traj = []
    T = max_steps or MAZE.step_limit
    for t in range(T):
        a_idx, probs = sample_action(s, w)
        if record:
            traj.append((s, a_idx, probs))
        s = advance(s, a_idx, MAZE)
        if s.done:
            break
    return s.score, traj if record else None

def eval_policy(w, episodes=8):
    scores = []
    for k in range(episodes):
        sc, _ = run_episode(w, seed=np.random.randint(0, 1<<31))
        scores.append(sc)
    return float(np.mean(scores)), float(np.std(scores))

## 6) Hill-Climbing Optimizer

In [ ]:
from dataclasses import dataclass

@dataclass
class HCConfig:
    sigma_init: float = 0.5
    sigma_min: float = 0.05
    decay: float = 0.995
    episodes_eval: int = 8
    iterations: int = 200

def hill_climb(w0, cfg: HCConfig):
    w = w0.copy()
    mean, std = eval_policy(w, episodes=cfg.episodes_eval)
    history = [(0, mean, std, cfg.sigma_init)]
    sigma = cfg.sigma_init
    best_w = w.copy(); best_mean = mean
    for it in range(1, cfg.iterations+1):
        eps = np.random.normal(0.0, sigma, size=w.shape)
        w_cand = w + eps
        m_cand, s_cand = eval_policy(w_cand, episodes=cfg.episodes_eval)
        if m_cand > mean:
            w = w_cand; mean, std = m_cand, s_cand
            if mean > best_mean:
                best_mean, best_w = mean, w.copy()
        sigma = max(cfg.sigma_min, sigma * cfg.decay)
        history.append((it, mean, std, sigma))
    return best_w, np.array(history)

w0 = np.random.normal(0, 0.1, size=FEAT_DIM)
cfg = HCConfig(sigma_init=0.6, sigma_min=0.05, decay=0.996, episodes_eval=6, iterations=160)
best_w, hist = hill_climb(w0, cfg)
print("Best mean score so far:", hist[-1,1])
print("Weights:", best_w)

## 7) Learning Curve

In [ ]:
fig = plt.figure(figsize=(6,4))
plt.plot(hist[:,0], hist[:,1])
plt.xlabel("Iteration"); plt.ylabel("Avg score (batch)")
plt.title("Hill-Climbing Learning Curve")
plt.tight_layout(); plt.show()

## 8) Rendering and Animation of a Sample Episode

In [ ]:
def render_state(ax, state: GameState, title=None):
    img = np.ones_like(MAZE.grid, dtype=float) * 0.9
    img[MAZE.grid==1] = 0.2
    for (i,j) in state.pellets: img[i,j] = 0.8
    for (i,j) in state.power:   img[i,j] = 0.5
    ax.imshow(img, interpolation='nearest')
    ax.scatter([state.pac[1]],[state.pac[0]], marker='o')
    if state.ghosts:
        ys = [g[0] for g in state.ghosts]; xs = [g[1] for g in state.ghosts]
        ax.scatter(xs, ys, marker='x')
    ax.set_axis_off()
    if title: ax.set_title(title)

score, traj = run_episode(best_w, record=True, seed=123)
print("Sample episode score:", score)

from matplotlib import animation
fig = plt.figure(figsize=(5,4.5)); ax = plt.gca()

def init_anim():
    s0 = traj[0][0]
    render_state(ax, s0, title="t=0")
    return []

def animate(k):
    ax.clear()
    s_k, a_k, _ = traj[k]
    render_state(ax, s_k, title=f"t={k}")
    return []

anim = animation.FuncAnimation(fig, animate, init_func=init_anim, frames=len(traj), interval=120, blit=False)
plt.close(fig)

try:
    from IPython.display import HTML
    HTML(anim.to_jshtml())
except Exception as e:
    print("Inline animation not available:", e)

## 9) Final Evaluation

In [ ]:
def evaluate_many(w, episodes=30):
    sc = []
    for k in range(episodes):
        s, _ = run_episode(w, seed=np.random.randint(0, 1<<31))
        sc.append(s)
    return float(np.mean(sc)), float(np.std(sc))

mean_final, std_final = evaluate_many(best_w, episodes=30)
print(f"Final policy average score over 30 episodes: {mean_final:.2f} ± {std_final:.2f}")

## 10) Save Artifacts & Download

In [ ]:
import os
os.makedirs("artifacts", exist_ok=True)

np.savez("artifacts/hc_pacman_run.npz",
         grid=MAZE.grid, start_pac=np.array(MAZE.start_pac),
         start_ghosts=np.array(MAZE.start_ghosts),
         power_pellets=np.array(MAZE.power_pellets),
         best_w=best_w, history=hist)

try:
    anim.save("artifacts/hc_pacman_episode.mp4")
    print("Saved animation to artifacts/hc_pacman_episode.mp4")
except Exception as e:
    print("Could not save MP4:", e)

print("Artifacts:", os.listdir("artifacts"))

In [ ]:
# Colab download helper
import shutil
from pathlib import Path
try:
    from google.colab import files  # type: ignore
    if Path('artifacts').exists():
        shutil.make_archive('artifacts', 'zip', 'artifacts')
        files.download('artifacts.zip')
    else:
        print('No artifacts folder found.')
except Exception as e:
    print('Colab download helper not available in this environment:', e)

## 11) Extensions & Experiments

- Add **simulated annealing** (accept worse candidates with small probability).
- Expand feature set: corridor width, ghost line‑of‑sight, pellet density ahead vs behind.
- Make ghosts smarter (ambush patterns) or add multiple difficulty modes.
- Swap hill‑climbing for **policy gradient REINFORCE** to compare sample efficiency.
- Add a reward for clearing all pellets quickly to shape behavior.